# Correlation-Pruned Fixed-Noise Sparse Bayesian Learning for Passive Intermodulation Cancellation

**Authors:** Oleg Kudryavtsev, Nikita Efimenko, and Anatoliy Efimov.

**Numerical example 1:** Clean cubic source with a cubic dictionary.

## Scope and reproducibility

This experiment evaluates passive intermodulation (PIM) cancellation with a nonlinear-order-matched dictionary. It generates the two transmit components, applies the common linear path and the phase-preserving cubic source, constructs the symmetry-reduced cubic Volterra dictionary, and compares dense ridge-regularized least squares with correlation-pruned fixed-noise sparse Bayesian learning (SBL).

The implementation includes all signal generation, model construction, estimation, evaluation, and generation of tables and figures. The likelihood precision is fixed within each SBL run and is varied only by the independent sweep. Execute the cells from top to bottom in a Python 3 environment with NumPy, SciPy, and Matplotlib.


In [ ]:
from __future__ import annotations

import csv
from dataclasses import dataclass
from itertools import combinations_with_replacement
from pathlib import Path
from time import perf_counter

import matplotlib.pyplot as plt
import numpy as np
from scipy.signal import firwin, lfilter

S = 5 * 2048
FS = 256e6
ORDER = 2048
F_SHIFT = 30e6
F_UP = 90e6
NL_GAIN = 1.2 + 3.7j
SEED = 12345

OUT = Path('out') / 'sbl_primary'
OUT.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({
    'figure.figsize': (11, 4.8),
    'figure.dpi': 110,
    'axes.grid': True,
    'axes.axisbelow': True,
    'grid.alpha': 0.3,
    'axes.spines.top': False,
    'axes.spines.right': False,
})

print(f'S = {S}, Fs = {FS / 1e6:g} MHz, FFT step = {FS / S / 1e3:.2f} kHz')


## 1. Two-carrier cubic PIM source

Let the low- and high-frequency transmit components be denoted by $x_\ell[n]$ and $x_h[n]$. After a shared causal linear path, the nonlinear source is

$$
g_3(s)=G\,s|s|^2,
$$

which is exactly cubic and preserves the phase of the complex envelope. The receiver processing frequency-shifts the selected $2f_\ell-f_h$ product to baseband and applies the known receiver low-pass filter. Both transmitter-side references and reference signals at the nonlinear source are retained, although the cancellation experiments use the transmitter-side references.


In [ ]:
@dataclass(frozen=True)
class StandaloneData:
    x_hi_tx: np.ndarray
    x_lo_tx: np.ndarray
    x_hi_at_pim: np.ndarray
    x_lo_at_pim: np.ndarray
    target_raw: np.ndarray
    target_filtered: np.ndarray
    lpf: np.ndarray


def generate_standalone_data(seed: int = SEED) -> StandaloneData:
    rng = np.random.RandomState(seed)
    s0 = rng.random_sample(S) + 1j * rng.random_sample(S)

    flt = firwin(ORDER + 1, 0.05, window='hamming', scale=True)
    n_fir = np.arange(ORDER + 1)
    flt_hi = flt * np.exp(1j * (2 * np.pi * F_SHIFT * n_fir / FS))
    flt_lo = flt * np.exp(1j * (2 * np.pi * -F_SHIFT * n_fir / FS))

    sig_hi = lfilter(flt_hi, [1.0], s0)
    sig_lo = lfilter(flt_lo, [1.0], s0)
    sig0 = sig_hi + sig_lo

    flt_in = np.array([
        0.186 - 0.023j, -0.098 + 0.117j, 0.291 - 0.338j, -0.607 + 0.628j,
        1.131 - 1.348j, -1.090 + 0.993j, 1.633 - 1.725j, -1.415 + 1.365j,
        0.760 - 0.668j, -0.256 + 0.211j, 0.045 - 0.037j,
    ])

    sig = lfilter(flt_in, [1.0], sig0)
    sig_nl = NL_GAIN * sig * np.abs(sig) ** 2

    n = np.arange(S)
    shifted = sig_nl * np.exp(1j * (2 * np.pi * F_UP * n / FS))
    lpf = firwin(ORDER + 1, 0.2, window='hamming', scale=True)
    target_filtered = lfilter(lpf, [1.0], shifted)

    rot_hi = np.exp(-1j * (2 * np.pi * F_SHIFT * n / FS))
    rot_lo = np.exp(1j * (2 * np.pi * F_SHIFT * n / FS))
    x_hi_tx = sig_hi * rot_hi
    x_lo_tx = sig_lo * rot_lo

    sig_hi_at_pim = lfilter(flt_in, [1.0], sig_hi)
    sig_lo_at_pim = lfilter(flt_in, [1.0], sig_lo)
    x_hi_at_pim = sig_hi_at_pim * rot_hi
    x_lo_at_pim = sig_lo_at_pim * rot_lo

    # The selected product is 2*(-30 MHz) - (+30 MHz) = -90 MHz, shifted to baseband by +90 MHz.
    target_raw = NL_GAIN * x_lo_at_pim * x_lo_at_pim * np.conj(x_hi_at_pim)

    return StandaloneData(
        x_hi_tx=x_hi_tx,
        x_lo_tx=x_lo_tx,
        x_hi_at_pim=x_hi_at_pim,
        x_lo_at_pim=x_lo_at_pim,
        target_raw=target_raw,
        target_filtered=target_filtered,
        lpf=lpf,
    )


data = generate_standalone_data()
print(f'target length = {len(data.target_filtered)}, lpf length = {len(data.lpf)}')


## 2. Cubic Volterra dictionary and dense ridge-LS baseline

For the causal delay set $D=\{0,\ldots,M-1\}$, the cubic candidates are

$$
\phi^{(3)}_{a,b,c}[n]
  =x_\ell[n-a]x_\ell[n-b]x_h^*[n-c],
  \qquad a\le b,\quad a,b,c\in D.
$$

The restriction $a\le b$ removes permutation duplicates without changing the span, giving

$$
N_3(M)=\binom{M+1}{2}M=\frac{M^2(M+1)}{2}.
$$

Every raw monomial is passed through the same receiver low-pass filter as the selected PIM observation. The dense reference estimator fits the full dictionary using the scale-adaptive ridge system

$$
\widehat{w}_{\mathrm{LS}}=(R+\lambda_X I)^{-1}r,
\quad R=\frac{X_f^H X_f}{N_f},\quad
r=\frac{X_f^H y_f}{N_f},\quad
\lambda_X=10^{-4}\frac{\operatorname{tr}(R)}{P}.
$$


In [ ]:
def lagged_matrix(x: np.ndarray, lags: list[int]) -> tuple[np.ndarray, np.ndarray]:
    """Rows contain x[n-lag] for all valid n and requested lags."""
    if not lags:
        raise ValueError('lags must not be empty')
    length = x.size
    start = max(0, max(lags))
    stop = min(length, length + min(lags))
    if stop <= start:
        raise ValueError('signal is too short for requested lags')
    n = np.arange(start, stop)
    return np.stack([x[n - lag] for lag in lags], axis=1), n


def im3_basis_lags(
    xa: np.ndarray,
    xb: np.ndarray,
    xc: np.ndarray,
    lags: list[int],
    *,
    symmetric: bool = True,
    selected_cols: np.ndarray | list[int] | None = None,
) -> tuple[np.ndarray, np.ndarray]:
    """Build xa(n-k1) * xb(n-k2) * conj(xc(n-k3))."""
    da, n_a = lagged_matrix(xa, lags)
    db, n_b = lagged_matrix(xb, lags)
    dc, n_c = lagged_matrix(xc, lags)
    if not (np.array_equal(n_a, n_b) and np.array_equal(n_a, n_c)):
        raise RuntimeError('lag grids are inconsistent')

    n_lags = len(lags)
    pairs = (
        list(combinations_with_replacement(range(n_lags), 2))
        if symmetric
        else [(k1, k2) for k1 in range(n_lags) for k2 in range(n_lags)]
    )
    n_cols = len(pairs) * n_lags
    selected = range(n_cols) if selected_cols is None else [int(col) for col in selected_cols]
    cols = []
    for col in selected:
        if col < 0 or col >= n_cols:
            raise ValueError(f'selected column {col} is outside 0..{n_cols - 1}')
        pair_idx, k3 = divmod(col, n_lags)
        k1, k2 = pairs[pair_idx]
        cols.append(da[:, k1] * db[:, k2] * np.conj(dc[:, k3]))
    if not cols:
        raise ValueError('selected_cols must not be empty')
    return np.stack(cols, axis=1), n_a


def filtered_basis(x_mat: np.ndarray, lpf: np.ndarray) -> np.ndarray:
    """Apply the known receiver LPF to each dictionary column."""
    return lfilter(lpf, [1.0], x_mat, axis=0)


def build_problem(
    data: StandaloneData,
    lags: list[int],
    *,
    use_oracle_refs: bool = False,
    apply_rx_lpf: bool = True,
    discard_samples: int = 2 * ORDER + 16,
    selected_cols: np.ndarray | list[int] | None = None,
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """Return X, y and absolute sample indices after transient removal."""
    if use_oracle_refs:
        x_lo, x_hi = data.x_lo_at_pim, data.x_hi_at_pim
        target = data.target_filtered if apply_rx_lpf else data.target_raw
    else:
        x_lo, x_hi = data.x_lo_tx, data.x_hi_tx
        target = data.target_filtered

    x_mat, n = im3_basis_lags(x_lo, x_lo, x_hi, lags, symmetric=True, selected_cols=selected_cols)
    if apply_rx_lpf:
        x_mat = filtered_basis(x_mat, data.lpf)

    y = target[n]
    start_by_abs_n = int(np.searchsorted(n, discard_samples))
    start_by_lpf = ORDER if apply_rx_lpf else 0
    start = max(start_by_abs_n, start_by_lpf)
    return x_mat[start:], y[start:], n[start:]


def cancellation_db(target: np.ndarray, residual: np.ndarray) -> float:
    eps = 1e-30
    return 10 * np.log10(np.mean(np.abs(target) ** 2) / (np.mean(np.abs(residual) ** 2) + eps))


def solve_wiener_hopf(x_mat: np.ndarray, y: np.ndarray, ridge: float = 1e-4) -> np.ndarray:
    k = x_mat.shape[0]
    r_mat = x_mat.conj().T @ x_mat / k
    r_vec = x_mat.conj().T @ y / k
    reg = ridge * np.trace(r_mat).real / r_mat.shape[0]
    return np.linalg.solve(r_mat + reg * np.eye(r_mat.shape[0]), r_vec)


## 3. Normalized correlation screening and fixed-noise SBL

On the fitting samples, the target and every nonzero dictionary column are normalized. A candidate receives the score

$$
R_i=\left|\bar{x}_{i,f}^{H}\bar{y}_f\right|,
$$

and is retained when $R_i>\eta$. If the threshold removes every candidate, the highest-scoring column is preserved. The screened normalized regression uses

$$
p(\bar{y}_f\mid v,\beta)=\mathcal{CN}(\bar{X}_f v,\beta^{-1}I),
\qquad
p(v\mid\alpha)=\mathcal{CN}(0,A^{-1}),
$$

with $A=\operatorname{diag}(\alpha_1,\ldots,\alpha_K)$. For fixed $\beta$,

$$
\Sigma=(\beta\bar{X}_f^H\bar{X}_f+A)^{-1},\qquad
\mu=\beta\Sigma\bar{X}_f^H\bar{y}_f,
$$

followed by $\gamma_i=1-\alpha_i\operatorname{Re}\{\Sigma_{ii}\}$ and $\alpha_i\leftarrow\gamma_i/(|\mu_i|^2+\varepsilon)$. Terms with non-finite or excessive precision are pruned. The surviving posterior means are finally rescaled to the original dictionary and target norms.


In [ ]:
@dataclass(frozen=True)
class SblConfig:
    beta: float = 100.0
    eta: float = 0.01
    keep_top: int | None = None
    min_keep: int = 1
    alpha_init: float = 1.0
    alpha_max: float = 1e6
    max_iter: int = 200
    tol: float = 1e-6
    eps: float = 1e-12


@dataclass(frozen=True)
class SblResult:
    h: np.ndarray
    active: np.ndarray
    screened: np.ndarray
    scores: np.ndarray
    alpha: np.ndarray
    mu_norm: np.ndarray
    iterations: int
    converged: bool
    y_norm: float
    col_norms: np.ndarray


def correlation_screen(
    x_normed: np.ndarray,
    y_normed: np.ndarray,
    *,
    eta: float,
    keep_top: int | None,
    min_keep: int,
) -> tuple[np.ndarray, np.ndarray]:
    scores = np.abs(x_normed.conj().T @ y_normed)
    keep = scores > eta

    if keep_top is not None:
        top = np.argsort(scores)[-min(keep_top, scores.size):]
        keep[top] = True

    if keep.sum() < min_keep:
        top = np.argsort(scores)[-min(min_keep, scores.size):]
        keep[:] = False
        keep[top] = True

    return np.flatnonzero(keep), scores


def solve_sbl_pruned(x_mat: np.ndarray, y: np.ndarray, cfg: SblConfig = SblConfig()) -> SblResult:
    if x_mat.ndim != 2:
        raise ValueError('x_mat must be 2-D')
    if y.ndim != 1 or y.size != x_mat.shape[0]:
        raise ValueError('y must be a vector with len(y) == x_mat.shape[0]')

    y_norm = float(np.linalg.norm(y))
    if y_norm <= cfg.eps:
        raise ValueError('target vector is numerically zero')

    col_norms = np.linalg.norm(x_mat, axis=0)
    nonzero_cols = np.flatnonzero(col_norms > cfg.eps)
    if nonzero_cols.size == 0:
        raise ValueError('all dictionary columns are numerically zero')

    x_normed = x_mat[:, nonzero_cols] / col_norms[nonzero_cols]
    y_normed = y / y_norm

    screened_local, scores_local = correlation_screen(
        x_normed,
        y_normed,
        eta=cfg.eta,
        keep_top=cfg.keep_top,
        min_keep=cfg.min_keep,
    )
    screened = nonzero_cols[screened_local]
    scores = np.zeros(x_mat.shape[1], dtype=float)
    scores[nonzero_cols] = scores_local

    x_active = x_normed[:, screened_local]
    active = screened.copy()
    alpha = np.full(x_active.shape[1], cfg.alpha_init, dtype=float)
    prev_mu: np.ndarray | None = None
    converged = False
    mu = np.zeros(x_active.shape[1], dtype=complex)

    for iteration in range(1, cfg.max_iter + 1):
        gram = x_active.conj().T @ x_active
        rhs = x_active.conj().T @ y_normed
        system = cfg.beta * gram + np.diag(alpha)

        sigma = np.linalg.inv(system)
        sigma = (sigma + sigma.conj().T) / 2
        mu = cfg.beta * sigma @ rhs

        gamma = 1.0 - alpha * np.real(np.diag(sigma))
        gamma = np.clip(gamma, cfg.eps, 1.0)
        alpha_new = gamma / (np.abs(mu) ** 2 + cfg.eps)
        alpha_new = np.clip(alpha_new, cfg.eps, 1e16)

        keep = np.isfinite(alpha_new) & (alpha_new <= cfg.alpha_max)
        if keep.sum() == 0:
            keep[np.argmin(alpha_new)] = True

        pruned = keep.sum() < keep.size
        if pruned:
            x_active = x_active[:, keep]
            active = active[keep]
            alpha = alpha_new[keep]
            mu = mu[keep]
            prev_mu = None
            continue

        if prev_mu is not None:
            denom = max(np.linalg.norm(prev_mu), cfg.eps)
            rel = np.linalg.norm(mu - prev_mu) / denom
            if rel < cfg.tol:
                alpha = alpha_new
                converged = True
                break

        prev_mu = mu.copy()
        alpha = alpha_new

    h = np.zeros(x_mat.shape[1], dtype=complex)
    h[active] = (y_norm / col_norms[active]) * mu
    return SblResult(
        h=h,
        active=active,
        screened=screened,
        scores=scores,
        alpha=alpha,
        mu_norm=mu,
        iterations=iteration,
        converged=converged,
        y_norm=y_norm,
        col_norms=col_norms,
    )


## 4. Numerical validation of the SBL implementation

Before the PIM experiment, a small complex-valued sparse regression problem verifies that the implementation identifies a compact support and reconstructs the target with low relative error. This diagnostic tests the inference routine independently of the PIM signal generator.


In [ ]:
def synthetic_sbl_check() -> None:
    rng = np.random.default_rng(7)
    n_rows, n_cols, n_active = 600, 80, 5
    x = (rng.standard_normal((n_rows, n_cols)) + 1j * rng.standard_normal((n_rows, n_cols))) / np.sqrt(2 * n_rows)
    h_true = np.zeros(n_cols, dtype=complex)
    support = rng.choice(n_cols, n_active, replace=False)
    h_true[support] = rng.standard_normal(n_active) + 1j * rng.standard_normal(n_active)
    y = x @ h_true
    y += 1e-3 * (rng.standard_normal(n_rows) + 1j * rng.standard_normal(n_rows)) / np.sqrt(2)

    res = solve_sbl_pruned(x, y, SblConfig(beta=1e4, eta=0.0, alpha_max=1e8, max_iter=300, tol=1e-7))
    pred = x @ res.h
    print(f'true support size = {n_active}, recovered active = {res.active.size}, iterations = {res.iterations}, converged = {res.converged}')
    print(f'fit quality = {cancellation_db(y, y - pred):.1f} dB')
    print('true support:', sorted(support.tolist()))
    print('top recovered:', sorted(res.active[np.argsort(np.abs(res.h[res.active]))[-n_active:]].tolist()))


synthetic_sbl_check()


## 5. Memory-depth experiment

The valid sample sequence is divided into fitting and held-out test portions. For each $M\in\{2,4,6,8\}$, ridge-LS is fitted on the full cubic dictionary, whereas SBL first applies one-shot correlation screening and then performs relevance pruning. Cancellation depth is evaluated as

$$
C=10\log_{10}\frac{\mathbb{E}|y|^2}{\mathbb{E}|y-\widehat{p}|^2}.
$$

The table also records the number of screened and active terms, the iteration count, the convergence flag, and implementation timings. The generated CSV contains the same run-level quantities.


In [ ]:
@dataclass(frozen=True)
class ExperimentRow:
    memory: int
    n_cols: int
    k_train: int
    k_test: int
    ls_train_db: float
    ls_test_db: float
    sbl_train_db: float
    sbl_test_db: float
    screened: int
    active: int
    iterations: int
    converged: bool
    ls_train_sec: float
    sbl_train_sec: float
    ls_infer_sec: float
    sbl_infer_sec: float


def run_memory_experiment(
    memory: int,
    *,
    sbl_cfg: SblConfig,
    ridge: float = 1e-4,
) -> tuple[ExperimentRow, np.ndarray, np.ndarray, np.ndarray, SblResult]:
    # Use the causal delay set D = {0, ..., M-1} defined for this experiment.
    lags = list(range(memory))
    x_mat, y, _ = build_problem(data, lags, use_oracle_refs=False, apply_rx_lpf=True)
    k_train = x_mat.shape[0] // 2
    x_train, y_train = x_mat[:k_train], y[:k_train]

    t0 = perf_counter()
    h_ls = solve_wiener_hopf(x_train, y_train, ridge=ridge)
    ls_train_sec = perf_counter() - t0

    t0 = perf_counter()
    sbl_res = solve_sbl_pruned(x_train, y_train, sbl_cfg)
    sbl_train_sec = perf_counter() - t0

    t0 = perf_counter()
    x_inf_ls, y_inf, _ = build_problem(data, lags, use_oracle_refs=False, apply_rx_lpf=True)
    pred_ls = x_inf_ls @ h_ls
    ls_infer_sec = perf_counter() - t0

    t0 = perf_counter()
    x_inf_sbl, y_sbl, _ = build_problem(
        data,
        lags,
        use_oracle_refs=False,
        apply_rx_lpf=True,
        selected_cols=sbl_res.active,
    )
    pred_sbl = x_inf_sbl @ sbl_res.h[sbl_res.active]
    sbl_infer_sec = perf_counter() - t0
    if y_inf.shape != y.shape or y_sbl.shape != y.shape:
        raise RuntimeError('inference timing used a different sample window')

    row = ExperimentRow(
        memory=memory,
        n_cols=x_mat.shape[1],
        k_train=k_train,
        k_test=x_mat.shape[0] - k_train,
        ls_train_db=cancellation_db(y[:k_train], y[:k_train] - pred_ls[:k_train]),
        ls_test_db=cancellation_db(y[k_train:], y[k_train:] - pred_ls[k_train:]),
        sbl_train_db=cancellation_db(y[:k_train], y[:k_train] - pred_sbl[:k_train]),
        sbl_test_db=cancellation_db(y[k_train:], y[k_train:] - pred_sbl[k_train:]),
        screened=sbl_res.screened.size,
        active=sbl_res.active.size,
        iterations=sbl_res.iterations,
        converged=sbl_res.converged,
        ls_train_sec=ls_train_sec,
        sbl_train_sec=sbl_train_sec,
        ls_infer_sec=ls_infer_sec,
        sbl_infer_sec=sbl_infer_sec,
    )
    return row, y, pred_ls, pred_sbl, sbl_res


def print_table(rows: list[ExperimentRow]) -> None:
    print(
        f"{'M':>3} {'N':>5} {'Ktr':>5} {'LS tr':>8} {'LS te':>8} "
        f"{'SBL tr':>8} {'SBL te':>8} {'screen':>7} {'active':>7} {'it':>4} "
        f"{'LS fit':>8} {'SBL fit':>8} {'LS inf':>8} {'SBL inf':>8}"
    )
    print('-' * 124)
    for r in rows:
        print(
            f'{r.memory:>3} {r.n_cols:>5} {r.k_train:>5} '
            f'{r.ls_train_db:>8.1f} {r.ls_test_db:>8.1f} '
            f'{r.sbl_train_db:>8.1f} {r.sbl_test_db:>8.1f} '
            f'{r.screened:>7} {r.active:>7} {r.iterations:>4} '
            f'{r.ls_train_sec:>8.3f} {r.sbl_train_sec:>8.3f} '
            f'{r.ls_infer_sec:>8.3f} {r.sbl_infer_sec:>8.3f}'
        )


# This configuration disables forced top-K retention and uses the correlation threshold directly.
sbl_cfg = SblConfig(beta=100.0, eta=0.01, keep_top=None, min_keep=1, alpha_max=1e6, max_iter=200, tol=1e-6)

rows = []
series = {}
for memory in (2, 4, 6, 8):
    row, y, pred_ls, pred_sbl, sbl_res = run_memory_experiment(memory, sbl_cfg=sbl_cfg)
    rows.append(row)
    series[memory] = (y, pred_ls, pred_sbl, sbl_res)

print_table(rows)

csv_path = OUT / 'sbl_pruned_standalone_results.csv'
with csv_path.open('w', newline='', encoding='utf-8') as f:
    writer = csv.writer(f)
    writer.writerow([
        'memory', 'n_cols', 'k_train', 'k_test', 'ls_train_db', 'ls_test_db',
        'sbl_train_db', 'sbl_test_db', 'screened', 'active', 'iterations', 'converged',
        'ls_train_sec', 'sbl_train_sec', 'ls_infer_sec', 'sbl_infer_sec',
    ])
    for r in rows:
        writer.writerow([
            r.memory, r.n_cols, r.k_train, r.k_test, r.ls_train_db, r.ls_test_db,
            r.sbl_train_db, r.sbl_test_db, r.screened, r.active, r.iterations, r.converged,
            r.ls_train_sec, r.sbl_train_sec, r.ls_infer_sec, r.sbl_infer_sec,
        ])

print(f'CSV saved to {csv_path}')


## 6. Frequency-domain diagnostic at the initial fixed precision

The spectrum compares the selected PIM component with the residuals produced by dense ridge-LS and sparse SBL. The plot is diagnostic: cancellation depth is computed in the time domain, while the spectrum shows how the residual energy is distributed across the receiver band.


In [ ]:
def spectrum_db(x: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    freqs = np.fft.fftshift(np.fft.fftfreq(x.size, 1.0 / FS)) / 1e6
    mag = 20 * np.log10(np.abs(np.fft.fftshift(np.fft.fft(x))) + 1e-300)
    return freqs, mag


best = max(rows, key=lambda r: r.sbl_test_db)
y, pred_ls, pred_sbl, sbl_res = series[best.memory]
residual_ls = y - pred_ls
residual_sbl = y - pred_sbl

freqs, before_db = spectrum_db(y)
_, ls_db = spectrum_db(residual_ls)
_, sbl_db = spectrum_db(residual_sbl)

fig, ax = plt.subplots(figsize=(11, 4.8))
ax.plot(freqs, before_db, lw=1.0, label='selected IM3 before cancellation')
ax.plot(freqs, ls_db, lw=1.0, label=f'LS residual, M={best.memory}, test={best.ls_test_db:.1f} dB')
ax.plot(freqs, sbl_db, lw=1.0, label=f'Pruned SBL residual, active={best.active}, test={best.sbl_test_db:.1f} dB')
ax.set_xlim(-28, 28)
ax.set_xlabel('Frequency, MHz')
ax.set_ylabel('20*log10|FFT(.)|, dB')
ax.set_title('Standalone PIM data: LS vs correlation-pruned fixed-noise SBL')
ax.legend()
fig.tight_layout()

plot_path = OUT / 'sbl_pruned_standalone.png'
fig.savefig(plot_path, dpi=130)
plt.show()

print(f'Best SBL: M={best.memory}, screened={best.screened}/{best.n_cols}, active={best.active}, iterations={best.iterations}')
print(f'Plot saved to {plot_path}')


## 7. Independent sweep over fixed likelihood precision

The likelihood precision $\beta$ is not updated by the SBL recursion. Instead, independent runs are performed for $\beta\in\{10^2,10^4,10^6\}$ and $M\in\{2,4,6,8\}$. Because the variables are normalized, $\beta$ is an algorithmic precision in normalized coordinates rather than an estimate of a physical receiver-noise level. Model selection should consider test cancellation, active support size, and convergence jointly.


In [ ]:
beta_rows = []
for beta in (100.0, 1e4, 1e6):
    for memory in (2, 4, 6, 8):
        cfg = SblConfig(beta=beta, eta=0.01, keep_top=None, min_keep=1, alpha_max=1e6, max_iter=500, tol=1e-6)
        row, *_ = run_memory_experiment(memory, sbl_cfg=cfg)
        beta_rows.append((beta, row))

print(
    f"{'beta':>9} {'M':>3} {'LS test':>8} {'SBL test':>9} {'active':>7} {'it':>4} {'conv':>5} "
    f"{'LS fit':>8} {'SBL fit':>8} {'LS inf':>8} {'SBL inf':>8}"
)
print('-' * 101)
for beta, row in beta_rows:
    print(
        f"{beta:>9.0g} {row.memory:>3} {row.ls_test_db:>8.1f} {row.sbl_test_db:>9.1f} "
        f"{row.active:>7} {row.iterations:>4} {str(row.converged):>5} "
        f"{row.ls_train_sec:>8.3f} {row.sbl_train_sec:>8.3f} "
        f"{row.ls_infer_sec:>8.3f} {row.sbl_infer_sec:>8.3f}"
    )

beta_csv_path = OUT / 'sbl_pruned_beta_sweep.csv'
with beta_csv_path.open('w', newline='', encoding='utf-8') as f:
    writer = csv.writer(f)
    writer.writerow([
        'beta', 'memory', 'n_cols', 'ls_test_db', 'sbl_test_db', 'screened', 'active',
        'iterations', 'converged', 'ls_train_sec', 'sbl_train_sec', 'ls_infer_sec', 'sbl_infer_sec',
    ])
    for beta, row in beta_rows:
        writer.writerow([
            beta, row.memory, row.n_cols, row.ls_test_db, row.sbl_test_db, row.screened, row.active,
            row.iterations, row.converged, row.ls_train_sec, row.sbl_train_sec,
            row.ls_infer_sec, row.sbl_infer_sec,
        ])

fig, ax = plt.subplots(figsize=(8.5, 4.8))
for memory in (2, 4, 6, 8):
    xs = [beta for beta, row in beta_rows if row.memory == memory]
    ys = [row.sbl_test_db for beta, row in beta_rows if row.memory == memory]
    ax.semilogx(xs, ys, 'o-', label=f'SBL M={memory}')
ls_m8 = next(row.ls_test_db for beta, row in beta_rows if beta == 100.0 and row.memory == 8)
ax.axhline(ls_m8, color='k', ls=':', lw=1.0, label=f'Ridge-LS M=8: {ls_m8:.1f} dB')
ax.set_xlabel('fixed beta')
ax.set_ylabel('Test suppression, dB')
ax.set_title('Fixed-noise SBL sensitivity to beta')
ax.legend()
fig.tight_layout()

beta_plot_path = OUT / 'sbl_pruned_beta_sweep.png'
fig.savefig(beta_plot_path, dpi=130)
plt.show()

best_beta, best_row = max(beta_rows, key=lambda item: item[1].sbl_test_db)
print(f'Best sweep result: beta={best_beta:g}, M={best_row.memory}, SBL={best_row.sbl_test_db:.1f} dB, active={best_row.active}')
print(f'CSV saved to {beta_csv_path}')
print(f'Plot saved to {beta_plot_path}')

best_cfg = SblConfig(beta=best_beta, eta=0.01, keep_top=None, min_keep=1, alpha_max=1e6, max_iter=500, tol=1e-6)
best_row, y_best, pred_ls_best, pred_sbl_best, best_sbl_res = run_memory_experiment(best_row.memory, sbl_cfg=best_cfg)

freqs, before_db = spectrum_db(y_best)
_, ls_best_db = spectrum_db(y_best - pred_ls_best)
_, sbl_best_db = spectrum_db(y_best - pred_sbl_best)

fig, ax = plt.subplots(figsize=(11, 4.8))
ax.plot(freqs, before_db, lw=1.0, label='selected IM3 before cancellation')
ax.plot(freqs, ls_best_db, lw=1.0, label=f'LS residual, M={best_row.memory}, test={best_row.ls_test_db:.1f} dB')
ax.plot(
    freqs,
    sbl_best_db,
    lw=1.0,
    label=f'Best pruned SBL residual, beta={best_beta:g}, active={best_row.active}, test={best_row.sbl_test_db:.1f} dB',
)
ax.set_xlim(-28, 28)
ax.set_xlabel('Frequency, MHz')
ax.set_ylabel('20*log10|FFT(.)|, dB')
ax.set_title('Selected fixed-noise SBL operating point versus ridge-LS')
ax.legend()
fig.tight_layout()

best_plot_path = OUT / 'sbl_pruned_best_beta.png'
fig.savefig(best_plot_path, dpi=130)
plt.show()
print(f'Selected-beta residual plot saved to {best_plot_path}')


## Interpretation of the selected operating points

With the deterministic configuration defined above, the highest SBL cancellation occurs at $M=6$ and $\beta=10^6$: approximately $51.3$ dB with seven active terms, compared with approximately $50.7$ dB for dense ridge-LS at the same memory. That SBL run reaches the iteration cap. A nearby converged point occurs at $M=8$ and $\beta=10^6$, with approximately $51.0$ dB and five active terms. These results illustrate the cancellation-complexity-convergence trade-off under finite causal memory.
